# Calculations and demos of the stereographic projection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import root

In [ ]:
def invert_x_minus_sin_x(y, *, method='hybr', tol=1e-06):
    if not np.asarray((y >= 0.0) & (y <= np.pi)).all():
        raise ValueError(f"y must be in [0, Pi], got y={y}")

    f = lambda x: x - np.sin(x) - y
    sol = root(f, x0=np.ones_like(y), method=method, tol=tol)
    if not sol.success:
        raise RuntimeError(f"Root finding failed: {sol.message}")
    return sol.x

In [ ]:
%%time
y = np.linspace(0, np.pi, 100)
x = invert_x_minus_sin_x(y, method='hybr', tol=1e-06)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=120)

ax.scatter(x, y, s=7**2, marker='x', label='hybr')

plt.show()

## Spherical binner

In [ ]:
from abc import ABC, abstractmethod

In [ ]:
class SphericalBinner(ABC):
    '''TODO
    '''
    @abstractmethod
    def r_limit(self, i: int):
        '''TODO'''
        raise NotImplementedError
    @staticmethod
    def r_centroid(self, i: int):
        '''TODO'''
        r0 = self.r_limit(i)
        r1 = self.r_limit(i+1)
        return self.centroid(r0, r1)
    @staticmethod
    def centroid(r0: float, r1: float):
        r'''
        Calculates the centroid of a conical frustum along the radius in
        3D space for the i-th bin between r0 and r1.
        The formula used is:
        .. math::
            r_i = \frac{1}{4} \frac{r_1^3 - r_0^3}{r_1^2 - r_0^2} + r_0
        where :math:`r_0` and :math:`r_1` are the lower and upper limits
        of the bin.

        Parameters:
        -----------
        r0 : float
            The lower limit of the bin.
        r1 : float
            The upper limit of the bin.

        Returns:
        --------
        r_i : float
            The centroid of the i-th bin.

        Notes:
        ------
        The formula is applied in a way to avoid numerical instability
        when :math:`r_0` and :math:`r_1` are very close to each other.
        '''
        nom = (r1-r0) * (r0*r0 + 2*r0*r1 + 3*r1*r1)  # r_1^3 - r_0^3
        den = (r0*r0 + r0*r1 + r1*r1)                # r_1^2 - r_0^2
        return 0.25 * nom / den + r0

    def invert_x_minus_sin_x(self, y, *, method='hybr', tol=1e-06):
        if not np.asarray((y >= 0.0) & (y <= np.pi)).all():
            raise ValueError(f"y must be in [0, Pi], got y={y}")

        f = lambda x: x - np.sin(x) - y
        sol = root(f, x0=np.ones_like(y), method=method, tol=tol)
        if not sol.success:
            raise RuntimeError(f"Root finding failed: {sol.message}")
        return sol.x

class SphericalLinear(SphericalBinner):
    r'''
    Radial binner that places equally spaced cuts in the stereographic
    polar half-angle :math:`\omega`.

    The first ``n_bins`` shells have an identical angular width
    :math:`\Delta\omega`. An optional fractional extension
    ``last_cell_size`` lets you append an extra (finite) zone so that
    the final grid point lies comfortably short of the projected
    "horizon" at :math:`\omega=\pi/2`.

    Parameters:
    -----------
    d_s : float
        The diameter of the 4D sphere.
    n_bins : int
        The number of radial bins to be used.
    last_cell_size : float
        The size of the last non-infinite cell.
    '''
    def __init__(self, d_s, n_bins, last_cell_size):
        self.d_s = d_s
        self.n_bins = n_bins
        self.last_cell_size = last_cell_size
        self.d_omega = np.pi / (2 * (self.n_bins + self.last_cell_size))

    def r_limit(self, i):
        omega = i * self.d_omega
        return self.d_s * np.tan(omega)

class SphericalConstantVolume(SphericalBinner):
    r'''
    Uniformly spaced shells in the Euclidean space with each shell
    enclosing the same Euclidean volume.

    The total volume inside the half-angle :math:`\omega` is given by
    .. math::
        V(\omega)
        =
        \int_0^{\omega} 4 \pi (d_s \tan(\omega'))^2
        \mathrm{d} (d_s \tan(\omega'))
        =
        2 \pi d_s^3 [ x - \sin(x) ]\,, \qquad x \equiv 2 \omega\,.


    Parameters:
    -----------
    d_s : float
        The diameter of the 4D sphere.
    n_bins : int
        The number of radial bins to be used.
    R_sim : float
        The radius of the simulation volume in real space.
    '''
    def __init__(self, d_s, n_bins, R_sim):
        self.d_s = d_s
        self.n_bins = n_bins
        # Largest possible half-angle, given ``r = d_s * tan(omega)``
        self.omega_max = 2 * np.arctan(R_sim/d_s)
        # Bin width in the Euclidean space, given 
        self.bin_width = (2*self.omega_max - np.sin(2*self.omega_max)) / n_bins

    def r_limit(self, i):
        omega = self.invert_x_minus_sin_x(i * self.bin_width) / 2  # or 1/4??
        return self.d_s * np.tan(omega)

In [ ]:
R_sim = 2000  # [Mph]
n_bins = 224
d_s = 105
last_cell_size = n_bins*np.pi / (2*np.arctan(R_sim/d_s)) - n_bins
print(f"last_cell_size = {last_cell_size}")

binner = SphericalLinear(d_s=d_s, n_bins=n_bins, last_cell_size=last_cell_size)
print(f"binner.d_omega = {binner.d_omega}")

binner = SphericalConstantVolume()

## Cylindrical binner

In [ ]:
class CylindricalBinner(ABC):
    '''TODO'''
    @abstractmethod
    def r_limit(self, i: int):
        '''TODO'''
        raise NotImplementedError
    @abstractmethod
    def r_centroid(self, i: int):
        '''TODO'''
        raise NotImplementedError
    @staticmethod
    def centroid(r0: float, r1: float):
        r'''
        Centroid of a cylindrical shell between r0 and r1 (assuming
        uniform height).
        The formula used is:
        .. math::
            r_i = \frac{2}{3} \frac{r_1^3 - r_0^3}{r_1^2 - r_0^2}
        where :math:`r_0` and :math:`r_1` are the lower and upper limits
        of the bin.
        '''
        return 2.0/3.0 * (r1*r1*r1 - r0*r0*r0) / (r1*r1 - r0*r0)

class CylindricalLinear(CylindricalBinner):
    '''TODO'''
    def __init__(self, R_max: float, n_bins: int):
        self.R_max = R_max
        self.n_bins = n_bins
        self.dr = R_max / n_bins

    def r_limit(self, i: int):
        return i * self.dr

    def r_centroid(self, i: int):
        r0 = self.r_limit(i)
        r1 = self.r_limit(i+1)
        # centroid of a cylindrical shell
        return self.centroid(r0, r1)

class CylindricalConstantVolume(CylindricalBinner):
    '''TODO'''
    def __init__(self, R_max: float, n_bins: int):
        self.R_max = R_max
        self.n_bins = n_bins
        self.dr = (R_max**2) / n_bins

    def r_limit(self, i: int):
        # invert A(r) = pi * r^2  ->  r_i = R_max * sqrt(i/n_bins)
        return np.sqrt(i * self.dr)

    def r_centroid(self, i: int):
        r0 = self.r_limit(i)
        r1 = self.r_limit(i+1)
        # centroid of a planar annulus
        return self.centroid(r0, r1)